# RBD analysis — Step 1a: Build the RBD whitelist

From the Gerstberger RBP/RBD census + the Pfam clans table, build a `family_name -> Pfam_acc` JSON used downstream by `parse_rbd_domtbl.py` to filter `hmmscan` hits.

**Inputs**
- `data/rbd/Gerstberger_RBP_list.xls`
- `data/rbd/Gerstberger_RBD_list.xlsx`
- `data/external/Pfam/Pfam-A.clans.tsv.gz`

**Output**
- `data/rbd/semiconservative_rbd2pfam.json`

Run from the repo root. The companion notebook `02_build_protein_fasta.ipynb` produces the FASTA consumed by `run_hmmscan.sh`.

In [1]:
import re
import json
import sys
from pathlib import Path

# Repo-root bootstrap: makes `coral` importable without installing the package.
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").is_file() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from coral import paths

import pandas as pd


In [5]:
df = pd.read_excel(paths.RBD_DIR / "Gerstberger_RBP_list.xls", sheet_name=1)
domain_names = df["domains[count]"].tolist()

In [6]:
cleaned_domain_names = list(set([re.sub(r'\[\d+\]', '', part) for item in domain_names for part in item.split(',')]))
cleaned_domain_names

['Ribosomal_S21e',
 'PSP',
 'EFG_IV',
 'PTRF_SDPR',
 'tRNA-synt_2c',
 'MPP6',
 'zf-RNPHF',
 'DUF1767',
 'zf-CXXC',
 'Ribosomal_L31e',
 'Methyltransf_4',
 'KOW',
 'EIF4E-T',
 'CwfJ_C_2',
 'Utp12',
 'PC4',
 'Ribosomal_S10',
 'RNA_pol_Rpb1_4',
 'PAZ',
 'DUF1387',
 'AKAP7_NLS',
 'DNA_RNApol_7kD',
 'Ribonuc_L-PSP',
 'Treacle',
 'Pept_tRNA_hydro',
 'Bin3',
 'Endonuclease_NS',
 'ADH_N',
 'MRP-S26',
 'DUF1891',
 'YjeF_N',
 'PROCN',
 'TPR_12',
 'PHD',
 'R3H',
 'LSM14',
 'PseudoU_synth_2',
 'Dus',
 'NLE',
 'Ribosomal_L24e',
 'S10_plectin',
 'Methyltrn_RNA_3',
 'RNase_Zc3h12a',
 'SRA1',
 'Med19',
 'Ribosomal_L19e',
 'RRS1',
 'FimP',
 'Nop16',
 'RNA_pol_Rpb4',
 'HGTP_anticodon',
 'Spt5-NGN',
 'mTERF',
 'PHAT',
 'VWA_3',
 'Metallophos',
 'EFG_II',
 'TGT',
 'MRP-S28',
 'Ribosomal_S5_C',
 'Npa1',
 'LRR_8',
 'Radical_SAM',
 'SNF2_N',
 'RNase_H2_suC',
 'DnaJ',
 'Nop',
 'U3_assoc_6',
 'CRM1_C',
 'BRCT',
 'eRF1_3',
 'MRP-L28',
 'HMG_box_5',
 'Ribosomal_L32p',
 'RNA_pol_L_2',
 'Mago_nashi',
 'UPF0086',
 '

In [8]:
rbd_df = pd.read_excel(paths.RBD_DIR / "Gerstberger_RBD_list.xlsx", sheet_name=0).rename({"Pfam RNA binding domains": "RBDs", "number of proteins in RBP census": "n_hits"}, axis=1)
rbd_names = rbd_df.loc[rbd_df["n_hits"] >= 4, "RBDs"].tolist()

clean_rbd_names = [dm for dm in cleaned_domain_names if dm in rbd_names]

/home/catalano/.conda/envs/encodersenv/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [ ]:
rbd_df

In [9]:
len(clean_rbd_names)

85

In [ ]:
len(cleaned_domain_names)

In [ ]:
clans = pd.read_csv(paths.PFAM_DIR / "Pfam-A.clans.tsv.gz", sep="\t", header=None,
                    names=["acc", "clan_acc", "clan_name", "pfam_name", "desc"])
name_to_acc = dict(zip(clans["pfam_name"], clans["acc"]))

accessions = {n: name_to_acc.get(n) for n in clean_rbd_names}
missing = [n for n, a in accessions.items() if a is None]
print(f"Resolved {len(accessions) - len(missing)} / {len(accessions)}")
print(f"Unresolved: {missing}")
for pid in missing:
    accessions.pop(pid)

In [ ]:
len(accessions)

### MANUALLY FINDING UNRESOLVED DOMAINS
Nol1_Nop2_Fmu has been renamed to Methyltr_RsmB-F (PF01189)<br>
APOBEC_N has been renamed to AID (PF08210). Beware of the incomplete description on InterPro (it only mentions the DNA-binding capabilities because that's what the AID proteins are more researched for, ignoring the foundational RNA-editing function of the ancestral domain (APOBEC1))<br>
RRM_6 has been merged into RRM_1 (PF00076)<br>
DZF does not exist anymore. There are DZF_N (PF07528) and DZF_C (PF20965) but their function is currently unknown<br>
HA2 has been renamed to WHD_HA2 (PF04408), we decided to include HA2_C (PF21010) as well<br>
PABP has been renamed to MLLE (PF00658)<br>
zf-RanBP has been renamed to Zn_ribbon_RanBP (PF00641)<br>

In [ ]:
# MANUALLY FINDING UNRESOLVED DOMAINS
unresolved_accessions = {"Methyltr_RsmB-F": "PF01189", "AID": "PF08210", "HA2_C": "PF21010", "WHD_HA2": "PF04408", "DZF_N": "PF07528", "MLLE": "PF00658", "Zn_ribbon_RanBP": "PF00641"}
accessions.update(unresolved_accessions)

In [ ]:
len(accessions)

In [ ]:
with open(paths.RBD_DIR / "semiconservative_rbd2pfam.json", "w") as f:
    json.dump(accessions, f, indent=2)